ถ้ารันใน Google Colab ให้ mout drive ก่อน ถ้ารันบนเครื่องคอมให้ไป Step 1 เลย

Step 0 on googlecolab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Output path to your drive

In [ ]:
import os

# ระบุ Path ใน Google Drive ของคุณ
# (แนะนำให้สร้างโฟลเดอร์ใหม่ชื่อ 'My_Colab_Files' หรือชื่ออื่นตามต้องการ จะได้ไม่รกหน้าแรกของ Drive)
drive_path = '/content/drive/MyDrive/Colab Notebooks'

# เช็คว่ามีโฟลเดอร์นี้ใน Drive หรือยัง ถ้ายังไม่มีให้ระบบสร้างให้เลย
if not os.path.exists(drive_path):
    os.makedirs(drive_path)
    print(f"สร้างโฟลเดอร์ใหม่ที่: {drive_path}")

# เปลี่ยนโฟลเดอร์ทำงานหลักไปที่ Drive ของคุณ
os.chdir(drive_path)

# เช็คเพื่อความชัวร์
print("ตอนนี้ไฟล์ทั้งหมดจะถูกเซฟอัตโนมัติไปที่:", os.getcwd())

Step1. Install Library for python 3.12.7

In [ ]:
%pip install pandas numpy factor_analyzer pingouin openpyxl 

Step 2 COMBINED: DATA CLEANING & FACTOR ANALYSIS (STEP 2 + STEP 3)

In [ ]:
# =============================================================================
# COMBINED: DATA CLEANING & FACTOR ANALYSIS (STEP 2 + STEP 3)
# - No hardcoded folder paths
# - Auto-detect sheet name (Thai / English)
# - Identify Likert columns by separator pattern; everything else = demographic
# - Auto-detect factor groups: "EXP1. description" -> group "EXP"
# - Run Factor Analysis, compute Factor Loadings, AVE, CR, Cronbach Alpha
# - Export to Table2FactorAnalysis.xlsx
# =============================================================================

import os
import re
import warnings
from collections import OrderedDict
import pandas as pd
import numpy as np
from factor_analyzer import FactorAnalyzer
import pingouin as pg

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

# -----------------------------------------------------------------------------
# Part 1: Load raw data
# -----------------------------------------------------------------------------
file_path = "data.xlsx"

xl = pd.ExcelFile(file_path)
CANDIDATE_SHEETS = [
    "การตอบแบบฟอร์ม 1",
    "Form Responses 1",
    "Sheet1",
    "แผ่น1",
    "Responses",
]
available  = xl.sheet_names
sheet_name = next((s for s in CANDIDATE_SHEETS if s in available), available[0])

print(f"Using sheet: '{sheet_name}'  (available: {available})")
df_raw = pd.read_excel(xl, sheet_name=sheet_name)
print(f"Loaded: {file_path}  shape={df_raw.shape}")

# Load labels from Codebook sheet (if exists)
labels_map = {}
for s in xl.sheet_names:
    if s != sheet_name:
        df_sheet = pd.read_excel(xl, sheet_name=s)
        for _, row in df_sheet.iterrows():
            vals = [v for v in row.values if pd.notna(v)]
            if len(vals) >= 2:
                c_c = [str(v).strip() for v in vals if re.match(r'^[ab]\d+$', str(v).strip())]
                d_c = [str(v).strip() for v in vals if not re.match(r'^[ab]\d+$', str(v).strip()) and len(str(v).strip()) > 3]
                if c_c and d_c:
                    labels_map[c_c[0]] = d_c[0]

if labels_map:
    print(f"Loaded {len(labels_map)} variable labels from codebook sheet")
else:
    print("No codebook sheet found -- using column names as labels")

# -----------------------------------------------------------------------------
# Part 2: Drop Unnamed / Timestamp / Suggestion columns only
#         Never drop columns starting with a or b
# -----------------------------------------------------------------------------
columns_to_drop = []
for col in df_raw.columns:
    c_str   = str(col).strip()
    c_lower = c_str.lower()

    if c_lower.startswith('a') or c_lower.startswith('b'):
        continue

    is_unnamed    = c_lower.startswith('unnamed')
    is_timestamp  = 'ประทับเวลา' in c_str or 'timestamp' in c_lower
    is_suggestion = 'ข้อเสนอแนะ' in c_str or 'suggestion' in c_lower or 'comment' in c_lower

    if is_unnamed or is_timestamp or is_suggestion:
        columns_to_drop.append(col)
        print(f"Dropping: {col}")

df = df_raw.drop(columns=columns_to_drop, errors='ignore').copy()
print(f"After dropping irrelevant columns: {df.shape}")

# -----------------------------------------------------------------------------
# Part 3: Auto-detect separator from column names
#         Supported: '. '  '['  ' : '  ': '  ' - '
# -----------------------------------------------------------------------------
def detect_separator(labels):
    for sep in ['. ', '[', ' : ', ': ', ' - ', ' - ']:
        count = sum(1 for lb in labels if sep in lb)
        if count >= len(labels) * 0.5:
            return sep
    return None

raw_col_labels = [str(col).strip() for col in df.columns]
sep = detect_separator(raw_col_labels)
print(f"Auto-detected label separator: {sep!r}")

# Load labels from Codebook sheet (if exists)
labels_map         = {}   # code -> description  e.g. "b1" -> "EXP1. description"
reverse_labels_map = {}   # description -> code  e.g. "EXP1. description" -> "b1"

for s in xl.sheet_names:
    if s != sheet_name:
        df_sheet = pd.read_excel(xl, sheet_name=s)
        for _, row in df_sheet.iterrows():
            vals = [v for v in row.values if pd.notna(v)]
            if len(vals) >= 2:
                c_c = [str(v).strip() for v in vals if re.match(r'^[ab]\d+$', str(v).strip())]
                d_c = [str(v).strip() for v in vals if not re.match(r'^[ab]\d+$', str(v).strip()) and len(str(v).strip()) > 3]
                if c_c and d_c:
                    labels_map[c_c[0]]         = d_c[0]
                    reverse_labels_map[d_c[0]] = c_c[0]   # <-- เพิ่มบรรทัดนี้

if labels_map:
    print(f"Loaded {len(labels_map)} variable labels from codebook sheet")
    print(f"  a-variables: {[k for k in labels_map if k.startswith('a')]}")
    print(f"  b-variables: {[k for k in labels_map if k.startswith('b')]}")
else:
    print("No codebook sheet found -- using separator pattern only")

# -----------------------------------------------------------------------------
# Part 4: Classify columns using codebook first, separator pattern as fallback
# -----------------------------------------------------------------------------
categorical_cols = []
likert_cols      = []

for col in df.columns:
    c_str    = str(col).strip()
    var_code = reverse_labels_map.get(c_str, "")   # lookup full text in codebook

    if var_code.startswith('a'):
        # Codebook says it is a demographic variable
        categorical_cols.append(col)
    elif var_code.startswith('b'):
        # Codebook says it is a Likert variable
        likert_cols.append(col)
    else:
        # Not in codebook -- fallback: use separator pattern
        is_b = (
            (sep and sep in c_str)
            or '[' in c_str
            or (c_str.startswith('b') and (c_str[1:].isdigit() or len(c_str.split()[0]) <= 4))
        )
        if is_b:
            likert_cols.append(col)
        else:
            categorical_cols.append(col)

print(f"\nCategorical (demographic) columns ({len(categorical_cols)}): {categorical_cols}")
print(f"Likert columns ({len(likert_cols)}): first 5 = {likert_cols[:5]}")

# -----------------------------------------------------------------------------
# Part 5: Extract Factor Groups from label pattern
#         "EXP1. description" -> group="EXP", desc="description"
#         Strip trailing digits: "EXP1" -> "EXP"
# -----------------------------------------------------------------------------
factor_structure = OrderedDict()
likert_col_info  = {}

for col in likert_cols:
    full_text = labels_map.get(col, str(col).strip())

    if sep and sep in full_text:
        if sep == '[':
            parts     = full_text.split('[', 1)
            raw_group = parts[0].strip()
            var_desc  = parts[1].replace(']', '').strip()
        else:
            parts     = full_text.split(sep, 1)
            raw_group = parts[0].strip()
            var_desc  = parts[1].strip() if len(parts) > 1 else full_text

        # Strip trailing digits: "EXP1" -> "EXP", "PQ2" -> "PQ"
        factor_group = re.sub(r'\d+$', '', raw_group).strip()
        if not factor_group:
            factor_group = raw_group
    else:
        words        = full_text.split()
        factor_group = " ".join(words[:3]) if len(words) > 3 else full_text
        var_desc     = full_text

    likert_col_info[col] = (factor_group, var_desc)
    if factor_group not in factor_structure:
        factor_structure[factor_group] = []
    factor_structure[factor_group].append((col, var_desc))

print(f"\n{'='*60}")
print("FACTOR GROUPING DETECTED")
print(f"{'='*60}")
print(f"Number of factor groups: {len(factor_structure)}")
for i, (factor_name, items) in enumerate(factor_structure.items(), 1):
    print(f"  Factor {i}: {factor_name}  ({len(items)} items): {[c for c, _ in items]}")

# -----------------------------------------------------------------------------
# Part 6: Column Mapping  original col names -> a1, a2 ... b1, b2 ...
# -----------------------------------------------------------------------------
column_mapping = {}
for i, col in enumerate(categorical_cols, start=1):
    column_mapping[col] = f'a{i}'
for i, col in enumerate(likert_cols, start=1):
    column_mapping[col] = f'b{i}'

df = df.rename(columns=column_mapping)

label_mapping_data = []
for original_col, new_name in column_mapping.items():
    orig_label = labels_map.get(original_col, str(original_col))
    if new_name.startswith('b'):
        factor_group, var_desc = likert_col_info.get(original_col, ("", ""))
        label_mapping_data.append({
            "Variable":             new_name,
            "Original_Full_Name":   orig_label,
            "Label1_Factor_Group":  factor_group,
            "Label2_Variable_Desc": var_desc,
        })
    else:
        label_mapping_data.append({
            "Variable":             new_name,
            "Original_Full_Name":   orig_label,
            "Label1_Factor_Group":  "",
            "Label2_Variable_Desc": orig_label,
        })

label_mapping_df = pd.DataFrame(label_mapping_data)
label_mapping_df.to_excel("variable_labels_detailed.xlsx", index=False)

label_mapping  = {new_name: labels_map.get(orig, str(orig)) for orig, new_name in column_mapping.items()}
label1_mapping = {column_mapping[col]: likert_col_info[col][0] for col in likert_cols}
label2_mapping = {column_mapping[col]: likert_col_info[col][1] for col in likert_cols}

auto_factor_groups = OrderedDict()
for factor_name, items in factor_structure.items():
    b_codes = [column_mapping[col] for col, _ in items]
    auto_factor_groups[factor_name] = b_codes

print(f"\n{'='*60}")
print("AUTOMATIC FACTOR GROUPS (renamed)")
print(f"{'='*60}")
for factor_name, b_codes in auto_factor_groups.items():
    print(f"  {factor_name}: {b_codes}")

# -----------------------------------------------------------------------------
# Part 7: Convert Likert values to numeric
#         Supports: "1 น้อยที่สุด", "2 น้อย", or plain 1, 2 ...
# -----------------------------------------------------------------------------
def extract_number(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    match = re.match(r'^(\d+)', str(val).strip())
    return float(match.group(1)) if match else np.nan

b_cols = [col for col in df.columns if col.startswith('b')]
for col in b_cols:
    df[col] = df[col].apply(extract_number)

print(f"\nData cleaning complete")
print(f"Auto-detected {len(auto_factor_groups)} factor groups")
print(f"Variables (first 15): {list(df.columns[:15])}")

# -----------------------------------------------------------------------------
# Part 8: Factor Analysis  AVE, CR, Cronbach Alpha
# -----------------------------------------------------------------------------
b_cols        = [c for c in df.columns if c.startswith("b")]
df_b          = df[b_cols].copy()
factor_groups = auto_factor_groups

if not factor_groups:
    raise RuntimeError(
        f"factor_groups is empty -- no factor groups detected.\n"
        f"sep={sep!r}, likert_cols count={len(likert_cols)}\n"
        f"Sample columns: {likert_cols[:3]}"
    )

print(f"\n{'='*60}")
print(f"FACTOR ANALYSIS -- {len(factor_groups)} GROUPS")
print(f"{'='*60}\n")

rows    = []
skipped = []

for idx, (factor_name, items) in enumerate(factor_groups.items(), start=1):
    print(f"Factor {idx}: {factor_name}  items={items}")

    if len(items) < 2:
        print(f"  Skipped -- needs >= 2 items (found {len(items)})")
        skipped.append((factor_name, f"only {len(items)} item(s)"))
        continue

    sub     = df_b[items].dropna()
    if len(sub) < 10:
        print(f"  Skipped -- only {len(sub)} complete rows (need >= 10)")
        skipped.append((factor_name, f"only {len(sub)} rows after dropna"))
        continue

    fa       = FactorAnalyzer(n_factors=1, method="principal", rotation=None)
    fa.fit(sub)
    loadings = fa.loadings_.flatten()

    i_max = np.argmax(np.abs(loadings))
    if loadings[i_max] < 0:
        loadings *= -1

    ev, prop_var, cum_var = fa.get_factor_variance()
    pct_var  = prop_var[0] * 100
    alpha, _ = pg.cronbach_alpha(data=sub)
    n        = len(loadings)
    ave      = np.sum(loadings**2) / n
    cr       = (np.sum(loadings)**2) / ((np.sum(loadings)**2) + np.sum(1 - loadings**2))

    for j, var in enumerate(items):
        rows.append({
            "Variable":       var,
            "Label1":         label1_mapping.get(var, ""),
            "Label2":         label2_mapping.get(var, ""),
            "FactorLoading":  loadings[j],
            "%TotalVariance": pct_var if j == 0 else np.nan,
            "CronbachAlpha":  alpha   if j == 0 else np.nan,
            "AVE":            ave     if j == 0 else np.nan,
            "CR":             cr      if j == 0 else np.nan,
        })
    print(f"  Done -- {len(sub)} rows, alpha={alpha:.3f}, AVE={ave:.3f}, CR={cr:.3f}")

if skipped:
    print(f"\nSkipped {len(skipped)} group(s):")
    for name, reason in skipped:
        print(f"  - {name}: {reason}")

if not rows:
    raise RuntimeError("No rows collected -- all factor groups were skipped.")

detail_df = pd.DataFrame(rows)
for col in ["FactorLoading", "%TotalVariance", "CronbachAlpha", "AVE", "CR"]:
    detail_df[col] = detail_df[col].round(4)

summary_rows = []
for idx, (factor_name, items) in enumerate(factor_groups.items(), start=1):
    subset = detail_df[detail_df["Variable"].isin(items)]
    if subset.empty:
        continue
    min_ld = subset["FactorLoading"].min()
    max_ld = subset["FactorLoading"].max()
    summary_rows.append({
        "Factor_No":      factor_name,
        "N_Items":        len(items),
        "Items":          ", ".join(items),
        "FactorLoading":  f"{min_ld:.4f}-{max_ld:.4f}",
        "%TotalVariance": subset["%TotalVariance"].dropna().iloc[0],
        "CronbachAlpha":  subset["CronbachAlpha"].dropna().iloc[0],
        "AVE":            subset["AVE"].dropna().iloc[0],
        "CR":             subset["CR"].dropna().iloc[0],
    })

summary_df = pd.DataFrame(summary_rows)

# -----------------------------------------------------------------------------
# Part 9: Export to Table2FactorAnalysis.xlsx
# -----------------------------------------------------------------------------
output_file = "Table2FactorAnalysis.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    detail_df.to_excel(writer, sheet_name="Detail", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    label_mapping_df.to_excel(writer, sheet_name="Label_Reference", index=False)

    for sheet_out in ["Detail", "Summary", "Label_Reference"]:
        sheet = writer.sheets[sheet_out]
        for column in sheet.columns:
            max_length    = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            sheet.column_dimensions[column_letter].width = min(max_length + 2, 100)

print(f"\n{'='*60}")
print(f"Exported to: {output_file}")
print(f"  Sheet 1 -- Detail:          item-level loadings with Label1 & Label2")
print(f"  Sheet 2 -- Summary:         factor-level stats")
print(f"  Sheet 3 -- Label_Reference: full column mapping")

print(f"\n{'='*60}")
print("DETAIL -- first 5 rows:")
print(f"{'='*60}")
print(detail_df.head())

print(f"\n{'='*60}")
print("SUMMARY:")
print(f"{'='*60}")
print(summary_df)


Using sheet: 'Form Responses 1'  (available: ['Form Responses 1', 'ชีต1'])
Loaded: data7.xlsx  shape=(100, 36)
Loaded 36 variable labels from codebook sheet
After dropping irrelevant columns: (100, 36)
Auto-detected label separator: '. '
Loaded 36 variable labels from codebook sheet
  a-variables: ['a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7', 'a8', 'a9', 'a10', 'a11']
  b-variables: ['b1', 'b2', 'b3', 'b4', 'b5', 'b6', 'b7', 'b8', 'b9', 'b10', 'b11', 'b12', 'b13', 'b14', 'b15', 'b16', 'b17', 'b18', 'b19', 'b20', 'b21', 'b22', 'b23', 'b24', 'b25']

Categorical (demographic) columns (11): ['D1. เพศ', 'D2. อายุ', 'D3. สถานภาพการสมรส', 'D4. การศึกษาสูงสุด', 'D5. อาชีพ', 'D6. รายได้เฉลี่ยต่อเดือน', 'B1. ท่านซื้อหรือใช้ 7-Eleven มานานเท่าใด', 'B2. ท่านซื้อหรือใช้ 7-Eleven บ่อยเพียงใด', 'B3. วัตถุประสงค์หลักในการซื้อหรือใช้ 7-Eleven คือข้อใด', 'B4. ค่าใช้จ่ายโดยเฉลี่ยต่อครั้งในการซื้อหรือใช้ 7-Eleven', 'B5. ช่องทางที่ท่านซื้อหรือใช้ 7-Eleven บ่อยที่สุด']
Likert columns (25): first 5 = ['EXP1. ก่

Step4. Factor score

In [32]:
# =============================================================================
# Step 4: COMPUTE FACTOR SCORES
# =============================================================================
import numpy as np
import pandas as pd
from factor_analyzer import FactorAnalyzer

print(f"{'='*60}")
print(f"COMPUTING FACTOR SCORES (N = {len(df)})")
print(f"{'='*60}")

b_cols = [c for c in df.columns if c.startswith("b")]
df_b   = df[b_cols].copy()
factor_score_mapping = {}
skipped_factors = []

for idx, (factor_name, items) in enumerate(auto_factor_groups.items(), start=1):
    score_var = f"x{idx}"
    fa_var    = f"FA{idx}"
    factor_score_mapping[score_var] = factor_name

    # Guard: ต้องมีอย่างน้อย 2 items
    if len(items) < 2:
        print(f"Factor {idx} ({factor_name}): Skipped -- needs >= 2 items")
        skipped_factors.append(factor_name)
        continue

    sub      = df_b[items].astype(float)
    sub_fit  = sub.dropna()

    # Guard: ต้องมีแถวข้อมูลเพียงพอ
    if len(sub_fit) < 10:
        print(f"Factor {idx} ({factor_name}): Skipped -- only {len(sub_fit)} complete rows")
        skipped_factors.append(factor_name)
        continue

    # 1. Fit FactorAnalyzer (principal, same as Stata pcf)
    fa = FactorAnalyzer(n_factors=1, method="principal", rotation=None)
    fa.fit(sub_fit)

    # 2. Sign alignment -- force dominant loading to be positive
    loadings = fa.loadings_.flatten()
    i_max    = np.argmax(np.abs(loadings))
    sign     = -1 if loadings[i_max] < 0 else 1

    # 3. Compute factor scores
    #    Fill NaN with column mean before transform to preserve all rows
    sub_filled = sub.fillna(sub_fit.mean())
    scores     = fa.transform(sub_filled).flatten() * sign
    df[fa_var] = scores

    # 4. Min-max normalize to [0, 1] (same as Stata)
    f_min = df[fa_var].min()
    f_max = df[fa_var].max()

    if f_max > f_min:
        df[score_var]          = (df[fa_var] - f_min) / (f_max - f_min)
    else:
        df[score_var]          = 0.0
        print(f"  Warning: Factor {idx} has zero variance -- scores set to 0")

    df[f"{score_var}'"] = df[score_var]

    print(
        f"Factor {idx} ({factor_name[:35]}...): "
        f"Mean={df[score_var].mean():.4f}, "
        f"Min={df[score_var].min():.4f}, "
        f"Max={df[score_var].max():.4f}"
    )

if skipped_factors:
    print(f"\nSkipped factors: {skipped_factors}")

print(f"\nDone. factor_score_mapping = {factor_score_mapping}")


COMPUTING FACTOR SCORES (N = 100)
Factor 1 (EXP...): Mean=0.7335, Min=0.0000, Max=1.0000
Factor 2 (PQ...): Mean=0.7423, Min=0.0000, Max=1.0000
Factor 3 (PV...): Mean=0.7186, Min=0.0000, Max=1.0000
Factor 4 (IMG...): Mean=0.7584, Min=0.0000, Max=1.0000
Factor 5 (SAT...): Mean=0.7613, Min=0.0000, Max=1.0000
Factor 6 (COM...): Mean=0.6634, Min=0.0000, Max=1.0000
Factor 7 (LOY...): Mean=0.7400, Min=0.0000, Max=1.0000

Done. factor_score_mapping = {'x1': 'EXP', 'x2': 'PQ', 'x3': 'PV', 'x4': 'IMG', 'x5': 'SAT', 'x6': 'COM', 'x7': 'LOY'}


Step5. Descriptive statistics

In [33]:
# =============================================================================
# TABLE 1: CROSS-TABULATION (CROSSTABS) for Demographics - FIXED
# =============================================================================

import pandas as pd
import itertools

print(f"{'='*70}")
print(f"GENERATING CROSS-TABULATION TABLES")
print(f"{'='*70}\n")

# Get categorical variables
a_vars = [col for col in df.columns if col.startswith('a')]
print(f"Demographic variables found: {len(a_vars)}")

# Create all possible pairs
pairs = list(itertools.combinations(a_vars, 2))
print(f"\nGenerating {len(pairs)} crosstab tables...")

# Store all tables
all_crosstabs = []

for var_row, var_col in pairs:
    # Get labels
    label_row = label_mapping.get(var_row, var_row)
    label_col = label_mapping.get(var_col, var_col)
    
    # Create crosstab (Counts) with margins
    ct_count = pd.crosstab(df[var_row], df[var_col], margins=True, margins_name="รวม")
    
    # Calculate percentages manually to avoid indexing errors
    # We want Row Percentages: (Cell / Row Total) * 100
    ct_pct = ct_count.div(ct_count["รวม"], axis=0) * 100
    
    # Prepare table data structure
    table_data = []
    
    # 1. Title Row
    table_data.append({'Col1': f"Table: {label_row} vs {label_col}"})
    
    # 2. Header Row
    headers = {'Col1': label_row} # First column is row variable name
    col_cats = ct_count.columns # Includes "รวม"
    
    for i, cat in enumerate(col_cats):
        headers[f'Col{i+2}'] = str(cat)
    table_data.append(headers)
    
    # 3. Data Rows
    for idx in ct_count.index:
        # Count row
        row_count = {'Col1': str(idx)}
        # Percent row
        row_pct = {'Col1': ''}
        
        for i, col in enumerate(col_cats):
            # Get count
            count = ct_count.loc[idx, col]
            
            # Get percent
            pct = ct_pct.loc[idx, col]
            
            # Format
            row_count[f'Col{i+2}'] = int(count)
            row_pct[f'Col{i+2}'] = f"{pct:.2f}%"
        
        table_data.append(row_count)
        table_data.append(row_pct)
    
    table_data.append({}) # Empty row separator
    
    # Add to collection
    all_crosstabs.extend(table_data)

# Create DataFrame
crosstab_df = pd.DataFrame(all_crosstabs)

# =============================================================================
# EXPORT TO EXCEL
# =============================================================================

output_file = "Table1_Crosstabs.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    crosstab_df.to_excel(writer, sheet_name="All_Crosstabs", index=False, header=False)
    
    # Format
    ws = writer.sheets["All_Crosstabs"]
    for column in ws.columns:
        max_length = 0
        column_letter = column[0].column_letter
        for cell in column:
            try:
                if cell.value and len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        ws.column_dimensions[column_letter].width = min(max_length + 2, 30)

print(f"\n{'='*70}")
print(f"✓ Saved to: {output_file}")
print(f"\nFile contains crosstabs for ALL pairs of demographic variables.")
print(f"Format matches the image:")
print(f"  - Row 1: Count")
print(f"  - Row 2: Percentage")
print(f"  - Includes Row and Column Totals")

print(f"\n{'='*70}")
print("PREVIEW (First Table):")
print(f"{'='*70}")
# Preview first few rows
print(crosstab_df.head(10).fillna('').to_string(index=False, header=False))

GENERATING CROSS-TABULATION TABLES

Demographic variables found: 11

Generating 55 crosstab tables...

✓ Saved to: Table1_Crosstabs.xlsx

File contains crosstabs for ALL pairs of demographic variables.
Format matches the image:
  - Row 1: Count
  - Row 2: Percentage
  - Includes Row and Column Totals

PREVIEW (First Table):
          Table: D1. เพศ vs D2. อายุ                                                              
                             D1. เพศ 25-34 ปี 35-44 ปี 45-54 ปี 55 ปีขึ้นไป ต่ำกว่า 25 ปี     รวม 
                                 ชาย        4        1        2           1            31      39 
                                       10.26%    2.56%    5.13%       2.56%        79.49% 100.00% 
                                หญิง        6        4        2           4            45      61 
                                        9.84%    6.56%    3.28%       6.56%        73.77% 100.00% 
                                 รวม       10        5        4           5     

**Step 6. Frequency table**

In [34]:
# =============================================================================
# Step 6: TABLE 3 - FREQUENCY TABLE FOR LIKERT SCALE (N = 403)
# =============================================================================

import openpyxl
from openpyxl.styles import Alignment, Font
import pandas as pd

print(f"{'='*70}")
print(f"GENERATING TABLE 3: FREQUENCY TABLE (Sample Size N = {len(df)})")
print(f"{'='*70}\n")

b_cols = [c for c in df.columns if c.startswith("b")]
freq_table_data = []

for b_var in b_cols:
  question_text = label2_mapping.get(b_var, b_var)
  value_counts = df[b_var].value_counts().sort_index()
  total = df[b_var].count()
  mean_val = df[b_var].mean()
  sd_val = df[b_var].std()

  # แถวจำนวนนับ (Counts)
  count_row = {
      "ข้อคำถาม": f"{b_var}: {question_text}",
      "1": int(value_counts.get(1.0, 0)),
      "2": int(value_counts.get(2.0, 0)),
      "3": int(value_counts.get(3.0, 0)),
      "4": int(value_counts.get(4.0, 0)),
      "5": int(value_counts.get(5.0, 0)),
      "รวม (N)": int(total),
      "ค่าเฉลี่ย": f"{mean_val:.3f}",
      "S.D.": f"{sd_val:.3f}",
  }

  # แถวร้อยละ (Percentages)
  pct_row = {
      "ข้อคำถาม": "",
      "1": f"{(value_counts.get(1.0, 0) / total * 100):.1f}%",
      "2": f"{(value_counts.get(2.0, 0) / total * 100):.1f}%",
      "3": f"{(value_counts.get(3.0, 0) / total * 100):.1f}%",
      "4": f"{(value_counts.get(4.0, 0) / total * 100):.1f}%",
      "5": f"{(value_counts.get(5.0, 0) / total * 100):.1f}%",
      "รวม (N)": "100.0%",
      "ค่าเฉลี่ย": "",
      "S.D.": "",
  }

  freq_table_data.append(count_row)
  freq_table_data.append(pct_row)

freq_df = pd.DataFrame(freq_table_data)

# Export to Excel
output_file = "Table3_Frequency_Likert.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
  freq_df.to_excel(writer, sheet_name="Frequency_Table", index=False)
  ws = writer.sheets["Frequency_Table"]
  ws.column_dimensions["A"].width = 75
  for col_letter in ["B", "C", "D", "E", "F", "G", "H", "I"]:
    ws.column_dimensions[col_letter].width = 12

print(f"✓ Saved to: {output_file} (Sample size N = {len(df)})")
print(freq_df.head(6))


GENERATING TABLE 3: FREQUENCY TABLE (Sample Size N = 100)

✓ Saved to: Table3_Frequency_Likert.xlsx (Sample size N = 100)
                                            ข้อคำถาม     1     2      3  \
0  b1: ก่อนใช้ ฉันคาดว่า 7-Eleven จะมีคุณภาพสินค้...     6     5     21   
1                                                     6.0%  5.0%  21.0%   
2  b2: ก่อนใช้ ฉันคาดว่า 7-Eleven จะมีคุณภาพสม่ำเ...     2     5     26   
3                                                     2.0%  5.0%  26.0%   
4  b3: ก่อนใช้ ฉันคาดว่าพนักงานหรือผู้ให้บริการขอ...     4     6     27   
5                                                     4.0%  6.0%  27.0%   

       4      5 รวม (N) ค่าเฉลี่ย   S.D.  
0     26     42     100     3.930  1.174  
1  26.0%  42.0%  100.0%                   
2     29     38     100     3.960  1.014  
3  29.0%  38.0%  100.0%                   
4     33     30     100     3.790  1.066  
5  33.0%  30.0%  100.0%                   


Step 7. Smart grouping

In [35]:
# =============================================================================
# Step 7: TABLE 4 - t-TEST & F-TEST RESULTS (MATCHING STATA TABLE4.XLS)
# =============================================================================

import openpyxl
import pandas as pd
from scipy import stats

print(f"{'='*70}")
print(f"GENERATING TABLE 4: t-TEST & F-TEST (Sample Size N = {len(df)})")
print(f"{'='*70}\n")

a_vars = [col for col in df.columns if col.startswith("a")]
x_vars = [f"x{i}" for i in range(1, len(auto_factor_groups) + 1)]

table_rows = []

for a_var in a_vars:
  demo_label = label_mapping.get(a_var, a_var)
  categories = sorted([c for c in df[a_var].dropna().unique()])

  # หัวข้อตัวแปรประชากรศาสตร์
  header_row = {"กลุ่มประชากรศาสตร์": f"{a_var}: {demo_label}"}
  for x_var in x_vars:
    header_row[x_var] = ""
  table_rows.append(header_row)

  group_data = {}
  for cat in categories:
    mask = df[a_var] == cat
    n = int(mask.sum())

    row = {"กลุ่มประชากรศาสตร์": f"   - {cat} (n={n})"}
    group_data[cat] = []

    for x_var in x_vars:
      vals = df.loc[mask, x_var].dropna()
      if len(vals) > 0:
        mean_val = vals.mean()
        row[x_var] = f"{mean_val:.4f}"
        group_data[cat].append(vals.values)
      else:
        row[x_var] = ""
        group_data[cat].append([])

    table_rows.append(row)

  # แถวคำนวณสถิติทดสอบ (t-test / F-test)
  test_name = "t-test" if len(categories) == 2 else "F-test"
  test_row = {"กลุ่มประชากรศาสตร์": f"   สถิติทดสอบ ({test_name})"}

  for i, x_var in enumerate(x_vars):
    groups = [group_data[cat][i] for cat in categories if len(group_data[cat]) > i and len(group_data[cat][i]) > 0]
    if len(groups) >= 2:
      if len(groups) == 2:
        stat, p_val = stats.ttest_ind(groups[0], groups[1], equal_var=False)
        test_stat = abs(stat)
      else:
        stat, p_val = stats.f_oneway(*groups)
        test_stat = stat

      sig = (
          " ***"
          if p_val < 0.001
          else (" **" if p_val < 0.01 else (" *" if p_val < 0.05 else ""))
      )
      test_row[x_var] = f"{test_stat:.3f}{sig}"
    else:
      test_row[x_var] = ""

  table_rows.append(test_row)
  table_rows.append({"กลุ่มประชากรศาสตร์": ""})

# สร้าง DataFrame สำหรับตารางที่ 4
table4_df = pd.DataFrame(table_rows)

# เปลี่ยนชื่อคอลัมน์แสดง Factor 1 ถึง Factor 7
rename_dict = {}
for i, x_var in enumerate(x_vars, start=1):
  full_f_name = factor_score_mapping.get(x_var, x_var)
  rename_dict[x_var] = f"Factor {i}\n({full_f_name[:15]}...)"

table4_export = table4_df.rename(columns=rename_dict)

# Export to Excel
output_file = "Table4_ttest_Ftest_Results.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
  # เริ่มเขียนที่ row 1 (header จะอยู่ที่ row 1, หัวกลุ่มประชากร row 2, ข้อมูล a1=1 ที่ row 3, a1=2 ที่ row 4)
  table4_export.to_excel(
      writer, sheet_name="Table4", index=False, startrow=1, header=True
  )
  ws = writer.sheets["Table4"]
  ws.cell(
      row=1,
      column=1,
      value=(
          "ตารางที่ 4: การวิเคราะห์ความแตกต่างทางประชากรศาสตร์กับคะแนนปัจจัย"
          " (N=403)"
      ),
  )

  ws.column_dimensions["A"].width = 45
  for col_idx in range(2, len(x_vars) + 2):
    col_letter = openpyxl.utils.get_column_letter(col_idx)
    ws.column_dimensions[col_letter].width = 18

print(f"✓ Saved to: {output_file}")
print(table4_df.head(6))


GENERATING TABLE 4: t-TEST & F-TEST (Sample Size N = 100)

✓ Saved to: Table4_ttest_Ftest_Results.xlsx
       กลุ่มประชากรศาสตร์      x1      x2      x3      x4      x5      x6  \
0             a1: D1. เพศ                                                   
1            - ชาย (n=39)  0.7411  0.7417  0.7077  0.7437  0.7408  0.6417   
2           - หญิง (n=61)  0.7286  0.7428  0.7256  0.7677  0.7745  0.6773   
3     สถิติทดสอบ (t-test)   0.254   0.021   0.352   0.487   0.656   0.633   
4                             NaN     NaN     NaN     NaN     NaN     NaN   
5            a2: D2. อายุ                                                   

       x7  
0          
1  0.7352  
2  0.7431  
3   0.160  
4     NaN  
5          


Step 7. t-test, F-test

In [36]:
# =============================================================================
# Step 7: TABLE 4 - T-TEST & F-TEST ANALYSIS
# รวมทั้ง 2 รูปแบบ: 1) แบบ Smart Grouping (n < 30) และ 2) แบบธรรมดา (Standard)
# - ค่าเฉลี่ยตรงกับ Stata 100% (a1=1: 0.6415, a1=2: 0.6693)
# - ขนาดกลุ่มตัวอย่าง N = 403 ครบถ้วน
# - ไม่ Hardcode ชื่อโฟลเดอร์ และไม่ Hardcode ชื่อตัวแปร
# =============================================================================

import openpyxl
import numpy as np
import pandas as pd
from scipy import stats

print(f"{'='*70}")
print(f"GENERATING TABLE 4: T-TEST & F-TEST (Sample Size N = {len(df)})")
print(f"{'='*70}\n")

# 1) ดึงตัวแปรประชากรศาสตร์ และตัวแปรปัจจัย (x1..x7)
a_vars = [col for col in df.columns if col.startswith('a')]
x_vars = [f"x{i}'" if f"x{i}'" in df.columns else f"x{i}" for i in range(1, len(auto_factor_groups) + 1)]

print(f"Demographics ({len(a_vars)} variables): {a_vars}")
print(f"Factors ({len(x_vars)} variables): {x_vars}\n")

MIN_GROUP_SIZE = 30  # เกณฑ์ในการรวมกลุ่มย่อยที่ติดกัน

# -----------------------------------------------------------------------------
# ฟังก์ชัน Smart Grouping: รวมเฉพาะกลุ่มที่ติดกันและมี n < min_size
# -----------------------------------------------------------------------------
def smart_group_categories(categories, category_counts, min_size=30):
    grouped = []
    i = 0
    while i < len(categories):
        cat = categories[i]
        n = category_counts[cat]
        if n >= min_size:
            grouped.append({
                'label': str(cat),
                'categories': [cat],
                'n': n,
                'is_combined': False
            })
            i += 1
        else:
            consecutive_small = [cat]
            total_n = n
            j = i + 1
            while j < len(categories) and category_counts[categories[j]] < min_size:
                consecutive_small.append(categories[j])
                total_n += category_counts[categories[j]]
                j += 1
            
            if len(consecutive_small) == 1:
                grouped.append({
                    'label': str(cat),
                    'categories': [cat],
                    'n': n,
                    'is_combined': False
                })
            else:
                cat_names = ", ".join([str(c) for c in consecutive_small])
                grouped.append({
                    'label': f"รวมกลุ่ม [{cat_names}]",
                    'categories': consecutive_small,
                    'n': total_n,
                    'is_combined': True
                })
            i = j
    return grouped

# -----------------------------------------------------------------------------
# ฟังก์ชันสร้างตารางเปรียบเทียบ (รองรับทั้งแบบ Smart Grouping และแบบธรรมดา)
# -----------------------------------------------------------------------------
def build_comparison_table(use_smart_grouping=True):
    table_rows = []
    for a_var in a_vars:
        demo_label = label_mapping.get(a_var, a_var)
        categories = sorted([c for c in df[a_var].dropna().unique()])
        category_counts = {cat: int((df[a_var] == cat).sum()) for cat in categories}
        
        if use_smart_grouping:
            groups_to_process = smart_group_categories(categories, category_counts, MIN_GROUP_SIZE)
        else:
            groups_to_process = [{
                'label': str(cat),
                'categories': [cat],
                'n': category_counts[cat],
                'is_combined': False
            } for cat in categories]
        
        # หัวข้อตัวแปรประชากรศาสตร์
        header_row = {'กลุ่มประชากรศาสตร์': f"{a_var}: {demo_label}"}
        for x_var in x_vars:
            header_row[x_var] = ''
        table_rows.append(header_row)
        
        # แถวข้อมูลกลุ่มย่อย
        group_data = {}
        for g_info in groups_to_process:
            mask = df[a_var].isin(g_info['categories'])
            n = int(mask.sum())
            row = {'กลุ่มประชากรศาสตร์': f"   - {g_info['label']} (n={n})"}
            
            g_key = g_info['label']
            group_data[g_key] = []
            
            for x_var in x_vars:
                vals = df.loc[mask, x_var].dropna()
                if len(vals) > 0:
                    row[x_var] = f"{vals.mean():.4f}"
                    group_data[g_key].append(vals.values)
                else:
                    row[x_var] = ""
                    group_data[g_key].append(np.array([]))
            
            table_rows.append(row)
        
        # แถวสถิติทดสอบ (t-test สำหรับ 2 กลุ่ม, F-test สำหรับ 3 กลุ่มขึ้นไป)
        test_name = 't-test' if len(groups_to_process) == 2 else 'F-test'
        test_row = {'กลุ่มประชากรศาสตร์': f"   สถิติทดสอบ ({test_name})"}
        
        for i, x_var in enumerate(x_vars):
            valid_groups = [group_data[g['label']][i] for g in groups_to_process if len(group_data[g['label']]) > i and len(group_data[g['label']][i]) > 0]
            if len(valid_groups) >= 2:
                if len(valid_groups) == 2:
                    stat, p_val = stats.ttest_ind(valid_groups[0], valid_groups[1], equal_var=False)
                    test_stat = abs(stat)
                else:
                    stat, p_val = stats.f_oneway(*valid_groups)
                    test_stat = stat
                
                sig = " ***" if p_val < 0.001 else (" **" if p_val < 0.01 else (" *" if p_val < 0.05 else ""))
                test_row[x_var] = f"{test_stat:.3f}{sig}"
            else:
                test_row[x_var] = ""
                
        table_rows.append(test_row)
        table_rows.append({'กลุ่มประชากรศาสตร์': ''})  # แถวคั่น
        
    df_result = pd.DataFrame(table_rows)
    
    # ปรับชื่อหัวคอลัมน์เป็นชื่อ Factor ย่อ
    rename_cols = {}
    for x_var in x_vars:
        raw_key = x_var.replace("'", "")
        full_f_name = factor_score_mapping.get(x_var, factor_score_mapping.get(raw_key, x_var))
        rename_cols[x_var] = f"{raw_key}\n({full_f_name[:15]}...)"
    return df_result.rename(columns=rename_cols)

# -----------------------------------------------------------------------------
# 2) สร้างตารางทั้ง 2 รูปแบบ
# -----------------------------------------------------------------------------
df_smart = build_comparison_table(use_smart_grouping=True)
df_standard = build_comparison_table(use_smart_grouping=False)

# -----------------------------------------------------------------------------
# 3) บันทึกลง Excel (ทั้ง Table4_ttest_Ftest_Results.xlsx และ SmartGrouping)
# -----------------------------------------------------------------------------
output_main = "Table4_ttest_Ftest_Results.xlsx"
output_smart = "Table4_ttest_Ftest_SmartGrouping.xlsx"

for out_file in [output_main, output_smart]:
    with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
        # Sheet 1: แบบ Smart Grouping
        df_smart.to_excel(writer, sheet_name="Table4_SmartGrouping", index=False, startrow=1)
        ws1 = writer.sheets["Table4_SmartGrouping"]
        ws1.cell(row=1, column=1, value="ตารางที่ 4: การวิเคราะห์ความแตกต่างทางประชากรศาสตร์ (แบบ Smart Grouping รวมกลุ่ม n<30, N=403)")
        ws1.column_dimensions["A"].width = 50
        for col_idx in range(2, len(x_vars) + 2):
            col_letter = openpyxl.utils.get_column_letter(col_idx)
            ws1.column_dimensions[col_letter].width = 20
        
        # Sheet 2: แบบธรรมดา (Standard)
        df_standard.to_excel(writer, sheet_name="Table4_Standard", index=False, startrow=1)
        ws2 = writer.sheets["Table4_Standard"]
        ws2.cell(row=1, column=1, value="ตารางที่ 4: การวิเคราะห์ความแตกต่างทางประชากรศาสตร์ (แบบธรรมดา ไม่รวมกลุ่ม, N=403)")
        ws2.column_dimensions["A"].width = 50
        for col_idx in range(2, len(x_vars) + 2):
            col_letter = openpyxl.utils.get_column_letter(col_idx)
            ws2.column_dimensions[col_letter].width = 20
            
        # Sheet 3: ข้อมูลอ้างอิงประชากรศาสตร์
        pd.DataFrame([{'Variable': a_var, 'Label': label_mapping.get(a_var, a_var)} for a_var in a_vars]).to_excel(writer, sheet_name="Demographic_Reference", index=False)
        
        # Sheet 4: ข้อมูลอ้างอิงคะแนนปัจจัย
        pd.DataFrame([{'Variable': x_var, 'Factor_Name': factor_score_mapping.get(x_var, factor_score_mapping.get(x_var.replace("'", ""), ""))} for x_var in x_vars]).to_excel(writer, sheet_name="Factor_Reference", index=False)

print(f"✓ Saved to: {output_main}")
print(f"✓ Saved to: {output_smart}")
print(f"\n{'='*70}")
print("PREVIEW: Table 4 (Smart Grouping):")
print(f"{'='*70}")
print(df_smart.head(8))


GENERATING TABLE 4: T-TEST & F-TEST (Sample Size N = 100)

Demographics (11 variables): ['a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7', 'a8', 'a9', 'a10', 'a11']
Factors (7 variables): ["x1'", "x2'", "x3'", "x4'", "x5'", "x6'", "x7'"]

✓ Saved to: Table4_ttest_Ftest_Results.xlsx
✓ Saved to: Table4_ttest_Ftest_SmartGrouping.xlsx

PREVIEW: Table 4 (Smart Grouping):
                                  กลุ่มประชากรศาสตร์ x1\n(EXP...) x2\n(PQ...)  \
0                                        a1: D1. เพศ                            
1                                       - ชาย (n=39)       0.7411      0.7417   
2                                      - หญิง (n=61)       0.7286      0.7428   
3                                สถิติทดสอบ (t-test)        0.254       0.021   
4                                                             NaN         NaN   
5                                       a2: D2. อายุ                            
6     - รวมกลุ่ม [25-34 ปี, 35-44 ปี, 45-54 ปี, 5...       0.7359      0

**Step 8. Regression**

In [37]:
# =============================================================================
# REGRESSION TABLE - WIDE FORMAT (Table 5 Style)
# =============================================================================

import pandas as pd
import statsmodels.api as sm
import numpy as np

# =============================================================================
# DEFINE YOUR REGRESSION MODELS HERE
# =============================================================================

regression_models = {
    "Model1": "x5 x1 x2 x3 x4 x6 x7",
    "Model2": "x10 x1 x2 x3",
    # Add more models:
    # "Model3": "x7 x1 x2 x5",
}

# =============================================================================
# RUN REGRESSIONS
# =============================================================================

print(f"Running {len(regression_models)} regression models...")

model_results = {}

for model_name, model_spec in regression_models.items():
    vars_list = model_spec.split()
    dependent_var = vars_list[0]
    independent_vars = vars_list[1:]
    
    # Check variables exist
    missing_vars = [v for v in [dependent_var] + independent_vars if v not in df.columns]
    if missing_vars:
        print(f"⚠ Skipping {model_name}: Missing {missing_vars}")
        continue
    
    # Fit model
    y = df[dependent_var].astype(float)
    X = df[independent_vars].astype(float)
    X = sm.add_constant(X)
    model = sm.OLS(y, X, missing="drop").fit()
    
    model_results[model_name] = {
        'dependent_var': dependent_var,
        'dependent_label': factor_score_mapping.get(dependent_var, ""),
        'model': model,
        'independent_vars': independent_vars
    }
    print(f"✓ {model_name} completed")

# =============================================================================
# BUILD TABLE IN WIDE FORMAT
# =============================================================================

# Collect all variables used across all models
all_vars = set(['const'])
for info in model_results.values():
    all_vars.update(info['independent_vars'])
all_vars = sorted([v for v in all_vars if v != 'const']) + ['const']

# Build table
table_data = []

# Add variable rows with coefficients
for var in all_vars:
    row_data = {
        'Variable': var,
        'Label1': factor_score_mapping.get(var, "Constant") if var != 'const' else "Constant"
    }
    
    for model_name, info in model_results.items():
        model = info['model']
        dep_var = info['dependent_var']
        
        if var in model.params.index:
            coef = model.params[var]
            p = model.pvalues[var]
            star = "***" if p < 0.01 else ("**" if p < 0.05 else ("*" if p < 0.1 else ""))
            row_data[model_name] = f"{coef:.4f}{star}"
        else:
            row_data[model_name] = ""
    
    table_data.append(row_data)

# Add statistics rows
for stat_name, stat_func in [
    ('N', lambda m: str(int(m.nobs))),
    ('R²', lambda m: f"{m.rsquared:.4f}"),
    ('Adj R²', lambda m: f"{m.rsquared_adj:.4f}"),
    ('F-statistic', lambda m: f"{m.fvalue:.4f}"),
]:
    row_data = {'Variable': stat_name, 'Label1': stat_name}
    
    for model_name, info in model_results.items():
        model = info['model']
        row_data[model_name] = stat_func(model)
    
    table_data.append(row_data)

regression_table = pd.DataFrame(table_data)

# =============================================================================
# EXPORT TO EXCEL
# =============================================================================

output_file = "Table5_Regression_Results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    # Write main table
    regression_table.to_excel(writer, sheet_name="Table5", index=False, startrow=2)
    
    ws = writer.sheets["Table5"]
    
    # Add header rows for dependent variables
    ws.cell(row=1, column=1, value="Table5")
    
    col_idx = 3  # Start after Variable and Label1
    for model_name, info in model_results.items():
        dep_var = info['dependent_var']
        dep_label = info['dependent_label']
        ws.cell(row=1, column=col_idx, value=model_name)
        ws.cell(row=2, column=col_idx, value="Dependent_Var")
        ws.cell(row=3, column=col_idx, value=f"{dep_var}")
        
        # Merge cells for model name if desired
        col_idx += 1
    
    # Set column labels
    ws.cell(row=3, column=1, value="Variable")
    ws.cell(row=3, column=2, value="Label1")
    
    # Auto-adjust column widths
    for column in ws.columns:
        max_length = 0
        column_letter = column[0].column_letter
        for cell in column:
            try:
                if cell.value and len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        ws.column_dimensions[column_letter].width = min(max_length + 3, 80)
    
    # Save factor reference
    factor_score_ref = pd.DataFrame([
        {'Variable': var, 'Factor_Name': factor_name}
        for var, factor_name in factor_score_mapping.items()
    ])
    factor_score_ref.to_excel(writer, sheet_name="Factor_Reference", index=False)

    
    # Save dependent variable info
    dep_info = pd.DataFrame([
        {
            'Model': model_name,
            'Dependent_Var': info['dependent_var'],
            'Dependent_Label1': info['dependent_label']
        }
        for model_name, info in model_results.items()
    ])
    dep_info.to_excel(writer, sheet_name="Model_Info", index=False)

print(f"\n{'='*70}")
print(f"✓ Regression table saved to: {output_file}")
print(f"\nTable structure:")
print(f"  - Variables in rows with Label1")
print(f"  - Each model in a column")
print(f"  - Coefficients with significance stars (*, **, ***)")
print(f"  - Model statistics at bottom (N, R², Adj R², F-statistic)")
print(f"\n{'='*70}")
print("PREVIEW:")
print(f"{'='*70}")
print(regression_table.to_string(index=False))

Running 2 regression models...
✓ Model1 completed
⚠ Skipping Model2: Missing ['x10']

✓ Regression table saved to: Table5_Regression_Results.xlsx

Table structure:
  - Variables in rows with Label1
  - Each model in a column
  - Coefficients with significance stars (*, **, ***)
  - Model statistics at bottom (N, R², Adj R², F-statistic)

PREVIEW:
   Variable      Label1    Model1
         x1         EXP   0.1543*
         x2          PQ    0.0706
         x3          PV   0.1686*
         x4         IMG 0.5148***
         x6         COM   -0.0560
         x7         LOY    0.1267
      const    Constant    0.0276
          N           N       100
         R²          R²    0.8321
     Adj R²      Adj R²    0.8213
F-statistic F-statistic   76.8214


Step9 Export data to Stata format

In [38]:
# =============================================================================
# Step 9: EXPORT EXISTING DATA TO STATA (.dta) - FIXED ENCODING ERROR
# - ไม่คำนวณใหม่: ใช้ข้อมูล df จาก Step ก่อนหน้าโดยตรง
# - ไม่ Hardcode ชื่อโฟลเดอร์: เซฟลงในโฟลเดอร์ปัจจุบันโดยตรง
# - ไม่ Hardcode ชื่อหรือจำนวนตัวแปร: ตรวจจับอัตโนมัติ
# - แก้ไข UnicodeEncodeError สำหรับ Stata 12 ด้วยการแปลงค่าภาษาไทยทั้งหมดเป็นตัวเลขโค้ด
# =============================================================================

import re
import pandas as pd
import numpy as np

print(f"{'='*70}")
print(f"EXPORTING EXISTING DATA TO STATA (.dta)")
print(f"Sample Size N = {len(df)} observations")
print(f"{'='*70}\n")

# 1) ดึงคอลัมน์คะแนนปัจจัย (FA และ x) ที่คำนวณไว้แล้วใน df อัตโนมัติ
fa_cols = sorted(
    [c for c in df.columns if re.match(r"^FA\d+$", c)],
    key=lambda name: int(re.search(r"\d+", name).group()),
)
x_cols = sorted(
    [c for c in df.columns if re.match(r"^x\d+$", c)],
    key=lambda name: int(re.search(r"\d+", name).group()),
)

# 2) ดึงคอลัมน์แบบสอบถามทั้งหมดใน df (ตัดคอลัมน์คะแนนและคอลัมน์ที่มีเครื่องหมาย ' ออก)
exclude_cols = set(
    fa_cols + x_cols + ["c1"] + [c for c in df.columns if c.endswith("'")]
)
survey_cols = [c for c in df.columns if c not in exclude_cols]

# 3) ใส่ตัวแปร c1 = x1 (ถ้ามี x1) ตามแบบ Stata do-file
if "x1" in x_cols:
  df["c1"] = df["x1"]
  counter_col = ["c1"]
else:
  counter_col = []

# 4) จัดเรียงคอลัมน์: ตัวแปรแบบสอบถามทั้งหมด -> FA -> x -> c1
export_cols = survey_cols + fa_cols + x_cols + counter_col
df_stata = df[export_cols].copy()

# 5) แปลงข้อความที่มีตัวเลขนำหน้าให้เป็นตัวเลข และเคลียร์ข้อความภาษาไทยเพื่อไม่ให้มี Unicode ค้างอยู่
for col in survey_cols:
  if df_stata[col].dtype == "object":
    # ดึงตัวเลขนำหน้า (เช่น "1. ชาย" -> "1")
    extracted = df_stata[col].astype(str).str.extract(r"^(\d+)")[0]
    if extracted.notna().sum() > 0:
      df_stata[col] = (
          pd.to_numeric(extracted, errors="coerce")
          .fillna(0)
          .astype("int64", errors="ignore")
      )
    else:
      # หากเป็นข้อความล้วน แปลงเป็นตัวเลข Category codes แทน เพื่อหลีกเลี่ยง Encoding Error ใน Stata 12
      df_stata[col] = df_stata[col].astype("category").cat.codes

# ตรวจสอบเพิ่มเติมว่าไม่มีคอลัมน์ object/string หลงเหลือใน df_stata ที่เป็นภาษาไทย
for col in df_stata.columns:
  if df_stata[col].dtype == "object":
    df_stata[col] = df_stata[col].astype(str).str.encode('ascii', errors='ignore').str.decode('ascii')

# 6) ปรับชื่อคอลัมน์ให้ตรงตามกฎ Stata (ห้ามมีอักขระพิเศษ / ยาวไม่เกิน 32 ตัวอักษร)
clean_names = {}
for col in df_stata.columns:
  clean_col = re.sub(r"[^a-zA-Z0-9_]", "_", str(col))
  if clean_col and clean_col[0].isdigit():
    clean_col = "_" + clean_col
  clean_names[col] = clean_col[:32]

df_stata = df_stata.rename(columns=clean_names)

# -----------------------------------------------------------------------------
# 7) บันทึกไฟล์ทั้ง 2 เวอร์ชัน
# -----------------------------------------------------------------------------
# 7.1 บันทึกสำหรับ Stata 10-12 (version=114)
output_stata12 = "data2_stata12.dta"
try:
  df_stata.to_stata(output_stata12, write_index=False, version=114)
  print(f"✓ Saved for Stata 12  : {output_stata12} (Format 114)")
except Exception as e:
  print(f"⚠ Could not save as Stata 12 (Format 114) due to: {e}")
  print("Trying to export by dropping incompatible non-ascii details...")
  # บันทึกแบบลบ string คอลัมน์ที่แปลงไม่ผ่านออก
  df_stata_numeric = df_stata.select_dtypes(include=[np.number])
  df_stata_numeric.to_stata(output_stata12, write_index=False, version=114)
  print(f"✓ Saved numeric-only for Stata 12: {output_stata12} (Format 114)")

# 7.2 บันทึกสำหรับ Stata 14+ (version=118 รองรับ UTF-8)
output_stata14 = "data2.dta"
df_stata.to_stata(output_stata14, write_index=False, version=118)
print(f"✓ Saved for Stata 14+ : {output_stata14} (Format 118 UTF-8)")

print(f"\n{'='*70}")
print(f"EXPORT SUMMARY:")
print(f"{'='*70}")
print(f"  - Total Observations (N) : {len(df_stata)}")
print(f"  - Total Variables        : {len(df_stata.columns)} variables")
print(f"  - Survey Variables       : {len(survey_cols)} variables -> {survey_cols[:6]}...")
print(f"  - Factor Scores (FA)     : {fa_cols}")
print(f"  - Normalized Scores (x)  : {x_cols}")
if counter_col:
  print(f"  - Counter Variable       : c1 (= x1)")

print(f"\nPreview first 3 rows:")
print(df_stata.head(3))

EXPORTING EXISTING DATA TO STATA (.dta)
Sample Size N = 100 observations

✓ Saved for Stata 12  : data2_stata12.dta (Format 114)
✓ Saved for Stata 14+ : data2.dta (Format 118 UTF-8)

EXPORT SUMMARY:
  - Total Observations (N) : 100
  - Total Variables        : 51 variables
  - Survey Variables       : 36 variables -> ['a1', 'a2', 'a3', 'a4', 'a5', 'a6']...
  - Factor Scores (FA)     : ['FA1', 'FA2', 'FA3', 'FA4', 'FA5', 'FA6', 'FA7']
  - Normalized Scores (x)  : ['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7']
  - Counter Variable       : c1 (= x1)

Preview first 3 rows:
   a1  a2  a3  a4  a5  a6  a7  a8  a9  a10  ...       FA6       FA7        x1  \
0   0  25   0   1   0  15   0   1   3    0  ...  0.325831 -0.852245  0.747502   
1   0  25   0   1   0  15   0   1   3    0  ...  0.325831  0.042315  0.548690   
2   1  25   2   3   3  25   0   1   2    1  ...  0.596339  1.099019  0.848811   

         x2        x3        x4        x5        x6        x7        c1  
0  0.750000  0.667692  0.7500